In [2]:
# Import the required Libraries
from IPython.display import Image
import torch
import os
import ultralytics
from ultralytics import YOLO
import onnx

# Check the version
torch.__version__
ultralytics.checks()

Ultralytics YOLOv8.2.103 🚀 Python-3.10.12 torch-2.8.0+cpu CPU (Intel Core(TM) i9-14900KS)
Setup complete ✅ (32 CPUs, 31.1 GB RAM, 1487.7/1832.2 GB disk)


In [3]:
HOME = os.getcwd()
print(HOME)

/home/aman/WorkSpace_STM32MP2/DMS/dms/Custom-DMS


In [4]:
!ls

camera_usb_gst.py	       mouth.py
dms_nb_models		       smoking_calling_yolov4.py
dms_tflite_models	       smoking_calling_yolov8n_custom.py
eye.py			       smoking_calling_yolov8.py
face_detection.py	       STM32_Dms-csi.py
face_detection_seperate_nb.py  STM32_Dms.py
face_landmark.py	       tests
model_convertion.ipynb	       yolov8n_int8.tflite


In [5]:
! /home/aman/stedgeai/install/2.2/Utilities/linux/stedgeai --help

usage: stedgeai --model FILE --target stm32|stellar-e|stellar-pg|ispu|mlc
                [--type keras|onnx|tflite] [--name STR]
                [--compression none|lossless|low|medium|high]
                [--allocate-inputs] [--allocate-outputs]
                [--no-inputs-allocation] [--no-outputs-allocation]
                [--input-memory-alignment INT] [--output-memory-alignment INT]
                [--workspace DIR] [--output DIR] [--split-weights]
                [--optimization OBJ] [--memory-pool FILE]
                [--no-onnx-optimizer] [--use-onnx-simplifier]
                [--fix-parametric-shapes FIX_PARAMETRIC_SHAPES]
                [--input-data-type float32|int8|uint8]
                [--output-data-type float32|int8|uint8]
                [--inputs-ch-position chfirst|chlast]
                [--outputs-ch-position chfirst|chlast]
                [--prefetch-compressed-weights] [--custom FILE]
                [--c-api st-ai|legacy] [--cut-input-tensors CUT_INPUT_

In [6]:
import os
os.environ["PATH"] += ":/home/aman/stedgeai/install/2.2/Utilities/linux"

In [7]:
# !stedgeai generate --model mobilefacenet_ir9_int8.onnx --target stm32mp35

!stedgeai generate \
  --model yolov8n_int8.tflite \
  --target stm32mp25 \
  --type tflite
  # --input-data-type int8 \
  # --output-data-type int8

ST Edge AI Core v2.2.0-20266 2adc00962
                                                                                
Model successfully compiled to NBG: /home/aman/WorkSpace_STM32MP2/DMS/dms/Custom-DMS/stm32ai_output/yolov8n_int8.nb
PASS: 	100%                                                                     
elapsed time (generate): 16.908s                                                


In [18]:
!stedgeai analyze \
  --model stm32ai_output/mobilefacenet_ir9.nb \
  --target stm32mp25 \
  --type onnx

ST Edge AI Core v2.2.0-20266 2adc00962
Analyse mode is not supported yet on STM32MP2
elapsed time (analyze): 0.326s


In [7]:
# this is to check if the model is really quantized or not if yes it will return True

import onnx
m = onnx.load("mobilefacenet.onnx")

print(any("QuantizeLinear" in n.op_type for n in m.graph.node))

False


In [19]:
import onnx
from onnx import mapping

model = onnx.load("mobilefacenet_ir9.onnx")

def get_dtype(tensor_type):
    return mapping.TENSOR_TYPE_TO_NP_TYPE.get(tensor_type.elem_type, "unknown")

print("=== INPUTS ===")
for inp in model.graph.input:
    t = inp.type.tensor_type

    shape = [d.dim_value for d in t.shape.dim]
    dtype = get_dtype(t)

    print(f"Name       : {inp.name}")
    print(f"Shape      : {shape}")
    print(f"Dtype      : {dtype}")
    print(f"Elem Type  : {t.elem_type}")
    print("-" * 40)

print("\n=== OUTPUTS ===")
for out in model.graph.output:
    t = out.type.tensor_type

    shape = [d.dim_value for d in t.shape.dim]
    dtype = get_dtype(t)

    print(f"Name       : {out.name}")
    print(f"Shape      : {shape}")
    print(f"Dtype      : {dtype}")
    print(f"Elem Type  : {t.elem_type}")
    print("-" * 40)

=== INPUTS ===
Name       : input
Shape      : [1, 3, 112, 112]
Dtype      : float32
Elem Type  : 1
----------------------------------------

=== OUTPUTS ===
Name       : embedding
Shape      : [1, 128]
Dtype      : float32
Elem Type  : 1
----------------------------------------
